# 7.1 Benchmarking Closed Models

Anthropic-only version of the closed-model benchmarking notebook.

Pipeline:
1. Configure parameters
2. Load dataset
3. Build prompts
4. Submit Anthropic batches
5. Poll status
6. Fetch + assemble predictions
7. Save outputs and evaluate

In [ ]:
# Parameters
PROVIDER = "anthropic"
API_KEY = None  # optional override; otherwise ANTHROPIC_API_KEY from env
MODEL_NAME = "claude-haiku-4-5"

DATASET_PATH = "./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased_categories_deduped.parquet"
OUTPUT_DIR = "./data/benchmark_results"
RUN_TAG = "closed_anthropic_run_01"
ANSWER_COL = "answer"
SAMPLE_SIZE = None

MAX_NEW_TOKENS = 20
USE_ANTHROPIC_BATCH = True
ANTHROPIC_BATCH_LIMIT = 100_000
BATCH_SIZE = 50  # used only if batch mode is disabled

EVENT_YEAR_FILTER = [year for year in range(2010, 2026)]
USE_INVARIANT_RELATIVE_CACHE = True
CACHE_SUBDIR = "invariant_relative_cache"
CHECKPOINT_EVERY = 5_000

PROMPT_TYPES = {
    "invariant": {"question_col": "question"},
    "relative": {"question_col": "relative_question"},
    "explicit": {
        "question_col": "question",
        "year_col": "event_year",
        "template": "In the year {year}, {question}",
    },
}
PROMPT_TYPES_YEAR_INDEPENDENT = [k for k, cfg in PROMPT_TYPES.items() if "year_col" not in cfg]
PROMPT_TYPES_YEAR_DEPENDENT = [k for k, cfg in PROMPT_TYPES.items() if "year_col" in cfg]

safe_model = MODEL_NAME.replace("/", "_").replace(".", "_")
assert PROVIDER == "anthropic"

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import ast
import json
import os
import re
import string
import time
import warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
client = Anthropic(api_key=API_KEY or os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
df = pd.read_parquet(DATASET_PATH)
if SAMPLE_SIZE:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

if EVENT_YEAR_FILTER is not None:
    if isinstance(EVENT_YEAR_FILTER, (list, set, tuple)):
        df = df[df["event_year"].isin(EVENT_YEAR_FILTER)].reset_index(drop=True)
    else:
        df = df[df["event_year"] == EVENT_YEAR_FILTER].reset_index(drop=True)

print(f"Loaded {len(df):,} rows")
display(df.head(2))

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant. Answer the question as concisely as "
    "possible. Provide only the answer phrase, with no explanation. "
    "EXAMPLE: Question: what is the color of the sky? Answer: blue"
)


def get_question_text(row, prompt_type: str) -> str:
    pcfg = PROMPT_TYPES[prompt_type]
    question = str(row[pcfg["question_col"]])
    if "year_col" in pcfg:
        year = row[pcfg["year_col"]]
        return pcfg["template"].format(year=int(year), question=question)
    return question


def _extract_anthropic_text(resp) -> str:
    for block in (resp.content or []):
        t = getattr(block, "text", None)
        if t is not None:
            return str(t)
    return ""


def _extract_text_from_batch_result(result) -> str:
    if not hasattr(result, "message") or not result.message or not result.message.content:
        return ""
    for block in result.message.content:
        t = getattr(block, "text", None)
        if t is not None:
            return str(t).strip().split("\n")[0].strip()
    return ""


def _call_api(question: str) -> str:
    q = str(question).strip()
    try:
        resp = client.messages.create(
            model=MODEL_NAME,
            max_tokens=MAX_NEW_TOKENS,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": q}],
        )
        return str(_extract_anthropic_text(resp)).strip().split("\n")[0].strip()
    except Exception:
        return ""


def run_batch_concurrent(questions: list[str], ptype: str) -> tuple[list[str], float]:
    predictions = [""] * len(questions)
    wall_start = time.time()
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as ex:
        futures = {ex.submit(_call_api, q): i for i, q in enumerate(questions)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"API [{ptype}]", unit="q"):
            idx = futures[fut]
            try:
                predictions[idx] = fut.result()
            except Exception:
                predictions[idx] = ""
    return predictions, time.time() - wall_start


def run_anthropic_batch(questions: list[str], custom_ids: list[str], ptype: str) -> str:
    if not questions:
        return ""
    batch_requests = []
    for cid, q in zip(custom_ids, questions):
        batch_requests.append(
            Request(
                custom_id=str(cid),
                params=MessageCreateParamsNonStreaming(
                    model=MODEL_NAME,
                    max_tokens=MAX_NEW_TOKENS,
                    system=SYSTEM_PROMPT,
                    messages=[{"role": "user", "content": str(q).strip()}],
                ),
            )
        )
    batch = client.messages.batches.create(requests=batch_requests)
    print(f"    Batch {batch.id} created ({ptype}, {len(questions):,} questions)")
    return batch.id


def fetch_anthropic_batch_results(batch_id: str) -> dict[str, str]:
    batch = client.messages.batches.retrieve(batch_id)
    if batch.processing_status != "ended":
        raise RuntimeError(f"Batch {batch_id} status: {batch.processing_status}. Not ready.")
    mapping = {}
    for result in client.messages.batches.results(batch_id):
        cid = result.custom_id
        if result.result.type == "succeeded":
            mapping[cid] = _extract_text_from_batch_result(result.result)
        else:
            mapping[cid] = ""
    return mapping

## Submit Anthropic Batches

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
dataset_stem = Path(DATASET_PATH).stem
cache_dir = Path(OUTPUT_DIR) / CACHE_SUBDIR
cache_dir.mkdir(parents=True, exist_ok=True)

all_predictions = {}
wall_times = {}
wall_start_total = time.time()
_BATCH_REGISTRY = {}
_BATCH_CONTEXT = {}

for ptype in PROMPT_TYPES:
    answer_col = f"model_answer_{ptype}"
    checkpoint_path = Path(OUTPUT_DIR) / f"checkpoint_{safe_model}_{RUN_TAG}_{ptype}.parquet"
    is_year_independent = ptype in PROMPT_TYPES_YEAR_INDEPENDENT

    if is_year_independent:
        key_col = PROMPT_TYPES[ptype]["question_col"]
        cache_path = cache_dir / f"cache_{ptype}_{safe_model}_{dataset_stem}.parquet"
        mapping = {}

        if USE_INVARIANT_RELATIVE_CACHE and cache_path.exists():
            cache_df = pd.read_parquet(cache_path)
            mapping = dict(zip(cache_df[key_col], cache_df[answer_col]))
            print(f"  [{ptype}] Loaded {len(mapping):,} from cache")

        keys = df.drop_duplicates(subset=[key_col])[key_col].tolist()
        remaining_keys = [k for k in keys if k not in mapping]
        remaining_questions = [str(k) for k in remaining_keys]

        if not remaining_questions:
            all_predictions[ptype] = ("mapping", mapping)
            wall_times[ptype] = 0.0
            print(f"  [{ptype}] Already complete")
            continue

        if USE_ANTHROPIC_BATCH:
            _BATCH_CONTEXT[ptype] = {
                "remaining_keys": remaining_keys,
                "existing_mapping": mapping,
                "key_col": key_col,
                "answer_col": answer_col,
                "cache_path": str(cache_path),
            }
            for chunk_start in range(0, len(remaining_questions), ANTHROPIC_BATCH_LIMIT):
                chunk_q = remaining_questions[chunk_start : chunk_start + ANTHROPIC_BATCH_LIMIT]
                custom_ids = [str(chunk_start + i) for i in range(len(chunk_q))]
                batch_id = run_anthropic_batch(chunk_q, custom_ids, ptype)
                _BATCH_REGISTRY[batch_id] = {
                    "provider": "anthropic",
                    "ptype": ptype,
                    "is_year_independent": True,
                    "chunk_start": chunk_start,
                    "chunk_size": len(chunk_q),
                }
        else:
            preds, elapsed = run_batch_concurrent(remaining_questions, ptype)
            for k, p in zip(remaining_keys, preds):
                mapping[k] = p
            if USE_INVARIANT_RELATIVE_CACHE:
                pd.DataFrame({key_col: list(mapping.keys()), answer_col: list(mapping.values())}).to_parquet(cache_path, index=False)
            all_predictions[ptype] = ("mapping", mapping)
            wall_times[ptype] = elapsed
    else:
        questions = [get_question_text(row, ptype) for _, row in df.iterrows()]
        resume_offset = 0
        preds_so_far = []

        if checkpoint_path.exists():
            ckpt_df = pd.read_parquet(checkpoint_path, columns=[answer_col])
            resume_offset = len(ckpt_df)
            preds_so_far = ckpt_df[answer_col].tolist()
            print(f"  [{ptype}] Resuming from checkpoint at {resume_offset:,}")

        remaining_q = questions[resume_offset:]
        if not remaining_q:
            all_predictions[ptype] = ("list", preds_so_far)
            wall_times[ptype] = 0.0
            continue

        if USE_ANTHROPIC_BATCH:
            _BATCH_CONTEXT[ptype] = {
                "answer_col": answer_col,
                "resume_offset": resume_offset,
                "preds_so_far": preds_so_far,
                "total_questions": len(questions),
                "checkpoint_path": str(checkpoint_path),
            }
            for chunk_start in range(0, len(remaining_q), ANTHROPIC_BATCH_LIMIT):
                chunk = remaining_q[chunk_start : chunk_start + ANTHROPIC_BATCH_LIMIT]
                global_offset = resume_offset + chunk_start
                custom_ids = [str(global_offset + i) for i in range(len(chunk))]
                batch_id = run_anthropic_batch(chunk, custom_ids, ptype)
                _BATCH_REGISTRY[batch_id] = {
                    "provider": "anthropic",
                    "ptype": ptype,
                    "is_year_independent": False,
                    "global_offset": global_offset,
                    "chunk_size": len(chunk),
                }
        else:
            preds = preds_so_far
            total_elapsed = 0.0
            offset = resume_offset
            while offset < len(questions):
                chunk_q = questions[offset : offset + CHECKPOINT_EVERY]
                preds_new, elapsed = run_batch_concurrent(chunk_q, ptype)
                preds.extend(preds_new)
                total_elapsed += elapsed
                offset += len(chunk_q)
                ckpt_df = df.iloc[: len(preds)].copy()
                ckpt_df[answer_col] = preds
                ckpt_df.to_parquet(checkpoint_path, index=False)
            if checkpoint_path.exists():
                checkpoint_path.unlink()
            all_predictions[ptype] = ("list", preds)
            wall_times[ptype] = total_elapsed

if _BATCH_REGISTRY:
    print(f"Submitted {len(_BATCH_REGISTRY)} Anthropic batch job(s).")
else:
    print("No batches submitted; sync path completed.")

In [ ]:
# Poll Anthropic batch status
if "_BATCH_REGISTRY" not in dir() or not _BATCH_REGISTRY:
    print("No pending batches.")
else:
    n_done, n_total = 0, len(_BATCH_REGISTRY)
    print(f"{'Batch ID':<40} {'Type':<12} {'Requests':>10} {'Status'}")
    print("-" * 80)
    for bid, meta in _BATCH_REGISTRY.items():
        batch = client.messages.batches.retrieve(bid)
        status = batch.processing_status
        done = status == "ended"
        print(f"{bid:<40} {meta['ptype']:<12} {meta['chunk_size']:>10,} {status}")
        if done:
            n_done += 1
    print(f"\n{n_done}/{n_total} complete")

In [ ]:
# Fetch completed Anthropic batches and assemble predictions
if "all_predictions" not in dir():
    all_predictions = {}
if "wall_times" not in dir():
    wall_times = {}
if "wall_start_total" not in dir():
    wall_start_total = time.time()

if "_BATCH_REGISTRY" not in dir() or not _BATCH_REGISTRY:
    print("No batches to fetch.")
else:
    fetched_results = {}
    skipped = 0

    for bid, meta in _BATCH_REGISTRY.items():
        batch = client.messages.batches.retrieve(bid)
        if batch.processing_status != "ended":
            print(f"Skipping {bid} ({meta['ptype']}) - {batch.processing_status}")
            skipped += 1
            continue
        results = fetch_anthropic_batch_results(bid)
        ptype = meta["ptype"]
        if ptype not in fetched_results:
            fetched_results[ptype] = {}
        for cid, text in results.items():
            fetched_results[ptype][int(cid)] = text

    for ptype in PROMPT_TYPES_YEAR_INDEPENDENT:
        if ptype not in fetched_results or ptype not in _BATCH_CONTEXT:
            continue
        ctx = _BATCH_CONTEXT[ptype]
        remaining_keys = ctx["remaining_keys"]
        mapping = dict(ctx["existing_mapping"])
        for pos, text in fetched_results[ptype].items():
            mapping[remaining_keys[pos]] = text

        cache_path = Path(ctx["cache_path"])
        key_col = ctx["key_col"]
        answer_col = ctx["answer_col"]
        pd.DataFrame({key_col: list(mapping.keys()), answer_col: list(mapping.values())}).to_parquet(cache_path, index=False)
        all_predictions[ptype] = ("mapping", mapping)
        wall_times[ptype] = 0.0

    for ptype in PROMPT_TYPES_YEAR_DEPENDENT:
        if ptype not in fetched_results or ptype not in _BATCH_CONTEXT:
            continue
        all_bids = [b for b, m in _BATCH_REGISTRY.items() if m["ptype"] == ptype]
        all_done = all(client.messages.batches.retrieve(b).processing_status == "ended" for b in all_bids)
        if not all_done:
            print(f"[{ptype}] Waiting for remaining chunks before assembling")
            continue

        ctx = _BATCH_CONTEXT[ptype]
        resume_offset = ctx["resume_offset"]
        total_questions = ctx["total_questions"]
        checkpoint_path = Path(ctx["checkpoint_path"])

        preds = list(ctx["preds_so_far"])
        for pos in range(resume_offset, total_questions):
            preds.append(fetched_results[ptype].get(pos, ""))

        all_predictions[ptype] = ("list", preds)
        wall_times[ptype] = 0.0
        if checkpoint_path.exists():
            checkpoint_path.unlink()

    wall_elapsed = time.time() - wall_start_total
    print(f"Fetch complete in {wall_elapsed / 60:.1f} min")
    if skipped:
        print(f"Skipped {skipped} incomplete batch(es)")

In [ ]:
# Attach predictions to dataframe and save
for ptype in PROMPT_TYPES:
    answer_col = f"model_answer_{ptype}"
    mode, payload = all_predictions[ptype]
    if mode == "mapping":
        key_col = PROMPT_TYPES[ptype]["question_col"]
        df[answer_col] = df[key_col].map(payload).fillna("")
    else:
        df[answer_col] = payload

output_path = Path(OUTPUT_DIR) / f"benchmark_{safe_model}_{RUN_TAG}.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved predictions -> {output_path}")

In [ ]:
def normalise(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def _as_list(val):
    if isinstance(val, list):
        return [str(v) for v in val]
    s = str(val).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            return [str(v) for v in p] if isinstance(p, list) else [s]
        except (ValueError, SyntaxError):
            pass
    return [s]


def exact_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) == pred for t in _as_list(truth)) else 0.0


def token_f1(pred: str, truth) -> float:
    pred_tokens = normalise(pred).split()
    truths = _as_list(truth)
    if not pred_tokens:
        return 0.0
    best = 0.0
    for t in truths:
        tt = normalise(t).split()
        if not tt:
            continue
        common = Counter(pred_tokens) & Counter(tt)
        n = sum(common.values())
        if n == 0:
            continue
        prec = n / len(pred_tokens)
        rec = n / len(tt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


def contains_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) in pred for t in _as_list(truth)) else 0.0


wall_elapsed = time.time() - wall_start_total
summary = {
    "model": MODEL_NAME,
    "provider": PROVIDER,
    "run_tag": RUN_TAG,
    "backend": "closed_api",
    "prompt_types": list(PROMPT_TYPES.keys()),
    "dataset_rows": len(df),
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "total_wall_time_min": round(wall_elapsed / 60, 2),
    "output_file": str(output_path),
}

for ptype in PROMPT_TYPES:
    col = f"model_answer_{ptype}"
    preds = df[col].fillna("").astype(str)
    truths = df[ANSWER_COL]
    summary[f"{ptype}_em"] = round(sum(exact_match(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_f1"] = round(sum(token_f1(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_contains"] = round(sum(contains_match(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_wall_min"] = round(wall_times.get(ptype, 0.0) / 60, 2)

summary_path = Path(OUTPUT_DIR) / f"benchmark_summary_{safe_model}_{RUN_TAG}.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))